# BR05A — Resolve rendering provenance and evaluate within generators

The source waveform can match even when PNG files differ. Do not delete the original evaluation gate or edit hashes. This notebook provides **two independent paths**:

1. Run native evaluation separately for each generator now. Each comparison then uses that generator's own recorded input images; it does not assert identical images across models.
2. Optionally reconstruct historical PULSE/newer PNGs, then compare their decoded pixels. This needs the two original rendering environments, but **no model loading or GPU inference**. It saves images in isolated directories, never overwriting shared query images. Only exact reconstruction of recorded PNG hashes followed by exact pixel equality permits the combined pixel-equivalence adapter.

If actual pixels differ, retain within-generator estimates and disclose the rendering difference. Model-to-model interactions are then confounded by rendering. A future definitive cross-model comparison needs one frozen set of input images with per-model provenance before inference.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
# BR04B generated reports in a context-fit CHILD run, not the original BR00 run.
# Leave None to select context_fit/v1/<cohort>/config.json from the base config.
# Or set the exact CHILD config path printed by BR04B.
CONTEXT_CONFIG_PATH = None
BASE_CONFIG_PATH = EXP / 'config.json'
if not BASE_CONFIG_PATH.is_file():
    raise FileNotFoundError('Missing experiment02/config.json; select the project used for BR04B')
BASE_CFG = load_config(BASE_CONFIG_PATH)
CONFIG_PATH = (Path(CONTEXT_CONFIG_PATH).expanduser() if CONTEXT_CONFIG_PATH is not None
               else Path(BASE_CFG['run_dir']) / 'context_fit' / 'v1' / BASE_CFG['cohort'] / 'config.json')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        'BR04B child config not found: ' + str(CONFIG_PATH)
        + '\nSet CONTEXT_CONFIG_PATH to the exact child config printed by BR04B. '
        + 'Do not substitute the original run config: its generation paths and packet hashes differ.')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Evaluation config:', CONFIG_PATH)
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from generation import folder_for
FOLDERS = {key: folder_for(CFG, key, False, 1024 if key == 'qwen38' or (CFG['cohort'] == 'shaoxing' and key == 'llama_base') else 384)
           for key in ('pulse', 'llama_ecg', 'llama_base', 'qwen38')}
# To evaluate an intentionally smaller generator set, edit FOLDERS before the first evaluation.
# Optional exact completed folders may be assigned here BEFORE the preflight below.
# Keep each folder from this SAME context-fit run and non-thinking protocol.
TARGETS_PATH = (Path(CFG['shaoxing_cohort']) / 'targets.json' if CFG['cohort'] == 'shaoxing'
                else Path(CFG['frozen_out']) / 'targets.json')
MAPPING_REVIEW_NOTE = ''  # REQUIRED for Shaoxing: document completed clinical mapping/annotation review.
ALLOW_PARTIAL = False  # Explicit True for a labeled matched-complete-case analysis if failures remain.
MAX_RUN_HOURS = 30.0
EVALUATION_VERSION = 'v1'
DESTINATION = RUN / 'evaluation' / CFG['cohort'] / EVALUATION_VERSION
REFERENCE_TEXTS = {}
if CFG['cohort'] != 'shaoxing':
    from report_study import reference_texts
    from queries import load_queries
    parent_config = read(PROJECT / 'config.json')
    text_cases = [dict(q, query_id=q['id']) for q in load_queries(CFG)]
    REFERENCE_TEXTS = reference_texts(PROJECT, parent_config, text_cases)
# Optional independent external report texts may be supplied; never synthesize narratives from codes.
print('Evaluating:', {k:str(v) for k,v in FOLDERS.items()})


## A. Evaluate each generator independently now

Use the CPU graph/evaluation environment. Run the next cell once for each MODEL_KEY. No rendering is performed. The original evaluator still checks that source hashes match across that generator's arms. Different per-model complete-case counts remain visible; do not present these as a common-query cross-model comparison. Retain all-arm and pairwise results and failure-sensitive F1.

In [ ]:
from evaluation import evaluate
MODEL_KEY = 'pulse'  # Then llama_ecg, llama_base, qwen38.
WITHIN_VERSION = 'within_generator_v1'
WITHIN_DESTINATION = RUN / 'evaluation' / CFG['cohort'] / WITHIN_VERSION / MODEL_KEY
result = evaluate(CFG, {MODEL_KEY: FOLDERS[MODEL_KEY]}, TARGETS_PATH, WITHIN_DESTINATION,
    reference_texts=REFERENCE_TEXTS, allow_partial=True,
    mapping_review_note=MAPPING_REVIEW_NOTE, max_run_hours=MAX_RUN_HOURS, repeats=2000)
print('Complete:', result['complete'])
if result['complete']:
    print('Matched queries for this generator:', result['matched_queries'])
    for row in result['summary']: print(row)
print('Within-generator outputs:', WITHIN_DESTINATION)


## B. Optional historical rendering reconstruction

Restart in the **original PULSE environment** first and run setup, folder definitions and the following reconstruction cell with RENDER_GROUP='pulse'. Then restart in the **original newer environment** and use RENDER_GROUP='modern'. Skip section A while in the old PULSE environment if its scoring dependencies are unavailable. No torch/Transformers upgrades, downloads or model loading are needed.

The helper checks exact NumPy/Pillow/Matplotlib versions against recorded generation protocols, source tensors, admission alignment and historical PNG file hashes. Matching versions alone does not establish reproduction (fonts/rc settings may differ). Stop if it cannot reproduce a historical hash. Internal Symile cohorts are supported; Shaoxing needs a separate reconstruction adapter.

Each saved image/check survives interruption. Keep the same audit directory to resume. First inspect a three-ECG pilot; then set MAX_NEW_RENDERS=0 for all 750 ECGs. Reproductions occupy additional disk space because their PNGs are retained for verification and review. Do not run simultaneous writers in one audit directory.

In [ ]:
from extensions.rendering_audit import reproduce
RENDER_GROUP = 'pulse'  # pulse in the PULSE environment; modern in the newer environment.
GROUP_MODELS = {'pulse': ('pulse',), 'modern': ('llama_ecg', 'llama_base', 'qwen38')}
MAX_NEW_RENDERS = 3  # Pilot first, then 0 to complete; successful reconstructions are skipped.
RENDER_AUDIT_VERSION = 'v1'
RENDER_ROOT = RUN / 'audits' / 'historical_rendering' / CFG['cohort'] / RENDER_AUDIT_VERSION
print(reproduce(CFG, {m:FOLDERS[m] for m in GROUP_MODELS[RENDER_GROUP]},
    RENDER_ROOT / RENDER_GROUP, max_new=MAX_NEW_RENDERS, max_run_hours=2.0))


## C. Compare only after both reconstructions finish

Run in the newer/CPU environment. Exact pixel equality is required for every ECG. If bytes reproduce but pixels differ, that is a real rendering difference; do not lower thresholds or use the combined evaluation below. Continue section A and report that limitation. This does not invalidate graph-versus-control comparisons within a generator when all its arms share the same image.

In [ ]:
from extensions.rendering_audit import compare
RENDER_AUDIT_VERSION = 'v1'
RENDER_ROOT = RUN / 'audits' / 'historical_rendering' / CFG['cohort'] / RENDER_AUDIT_VERSION
comparison = compare(RENDER_ROOT / 'pulse', RENDER_ROOT / 'modern', RENDER_ROOT / 'pixel_comparison.json')
print({k:v for k,v in comparison.items() if k != 'queries'})
print('Differing cases:', [r['query_id'] for r in comparison['queries'] if not r['identical_pixels']][:10])


## D. Combined evaluation ONLY if exact historical pixels match

This verifies both audits again and validates all native report provenance. It uses a canonical pixel identity for the in-memory source check while leaving all original journals and protocols untouched. A separate immutable equivalence protocol records the evidence. There is no manual waiver. If any pixel differs, this cell stops. Use a new evaluation directory, not the failed v1 directory.

This does not automatically repair human-review image matching. Before BR07, supply the reviewer images whose hashes match the chosen models' historical inputs; do not rerender shared images in an arbitrary environment. If encodings alone differ, the clinical helper still needs an explicit pixel-equivalence-aware extension before combining review models; do not bypass its checks.

In [ ]:
from extensions.rendering_audit import evaluate_pixel_equivalent
COMBINED_DESTINATION = RUN / 'evaluation' / CFG['cohort'] / 'v2_exact_pixel_equivalence'
result = evaluate_pixel_equivalent(CFG, FOLDERS, TARGETS_PATH, COMBINED_DESTINATION,
    RENDER_ROOT / 'pulse', RENDER_ROOT / 'modern', reference_texts=REFERENCE_TEXTS,
    allow_partial=True, mapping_review_note=MAPPING_REVIEW_NOTE,
    max_run_hours=MAX_RUN_HOURS, repeats=2000)
print('Complete:', result['complete'])
if result['complete']: print('Common matched queries:', result['matched_queries'])
print('Outputs:', COMBINED_DESTINATION)
